# 09. 딥러닝 이진분류 - 실무전용 문제 모음

> **실무 전용 노트북** - 이론 설명 없이 TODO 문제만 모았습니다. (관련 이론 노트북: 09_딥러닝_분류_이진.ipynb)
이미 개념은 이해했다는 전제로, 손으로 더 많이 풀어보며 완전히 몸에 익히는 것이 목표입니다. 정답을 먼저 보지 말고 반드시 스스로 코드를 작성해본 뒤 펼쳐서 비교하세요.

이론 노트북에서는 Titanic으로 연습했다면, 여기서는 **유방암 진단(breast_cancer) 데이터**로 딥러닝 이진분류를 다시 연습합니다.

## 목차
1. 모델 설계 연습 (Dropout 포함)
2. 모델 학습 연습
3. 성능 평가 연습

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

tf.random.set_seed(100)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('../data/breast_cancer.csv')
X = df.drop(columns=['target'])
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
X_train.shape

## 1. 모델 설계 연습 (Dropout 포함)

**문제 1.** 입력층 32(relu)+Dropout(0.3)→16(relu)→출력 1(sigmoid)인 `model`을 만들어 `binary_crossentropy`로 컴파일하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
model = Sequential()
model.add(Dense(32, activation='relu', input_shape=(X_train_s.shape[1],)))
model.add(Dropout(0.3))  # 30%의 노드 연결을 무작위로 꺼서 과적합을 줄임(추론 시에는 적용되지 않음)
model.add(Dense(16, activation='relu'))
model.add(Dense(1, activation='sigmoid'))  # 이진분류 출력층: 노드 1개 + sigmoid
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()
```

</details>

## 2. 모델 학습 연습

**문제 2.** `EarlyStopping(patience=10)`로 `epochs=100`까지 학습시키세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from tensorflow.keras.callbacks import EarlyStopping
es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)  # restore_best_weights=True: 학습이 끝나면 가장 좋았던 시점의 가중치로 되돌림
history = model.fit(X_train_s, y_train, epochs=100, batch_size=16, validation_data=(X_test_s, y_test), callbacks=[es], verbose=0)
print(len(history.history['loss']))
```

</details>

## 3. 성능 평가 연습

**문제 3.** 예측 확률을 0.5 기준으로 0/1로 바꿔 accuracy와 recall을 구하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.metrics import accuracy_score, recall_score
pred_proba = model.predict(X_test_s).flatten()  # 확률(0~1) 그대로는 정확도를 계산할 수 없어 flatten으로 1차원 배열로 변환
pred_label = (pred_proba > 0.5).astype(int)  # 0.5 임계값으로 확률을 0/1 클래스로 변환
print(accuracy_score(y_test, pred_label), recall_score(y_test, pred_label))
```

</details>

**문제 4.** confusion matrix를 heatmap으로 그리세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, pred_label)
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds')  # annot=True로 칸 안에 숫자를 함께 표시
plt.show()
```

</details>